# Sign Language Translation — End-to-End Notebook

ASL video → English text · ResNet-18 + Landmark-Aware Spatial Attention + Transformer

| Split | Role |
|---|---|
| **train** | Model training (phases 1 + 2) |
| **val** | Per-epoch BLEU monitoring, early stopping |
| **test** | Final held-out evaluation — touched once, after training |

---
**Sections**
1. Environment & GPU check
2. Configuration
3. Data pipeline walkthrough (frame extraction · landmarks · heatmaps)
4. Tokenizer
5. Dataset & DataLoader — all three splits
6. Model architecture
7. Training (train split, val monitoring)
8. Test-set evaluation (BLEU-4 · ROUGE-L · METEOR)
9. LLM-as-Judge on test predictions
10. Ablation study on test split
11. Save & load checkpoint
12. Interactive inference

## 1. Environment & GPU Check

In [ ]:
import sys, io, time, json, logging
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch

ROOT = Path(".").resolve()
sys.path.insert(0, str(ROOT))

logging.basicConfig(level=logging.WARNING)

print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU     : {props.name}")
    print(f"VRAM    : {props.total_memory / 1e9:.1f} GB")
    print(f"CUDA    : {torch.version.cuda}")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nActive device : {DEVICE}")

## 2. Configuration

Set `DEMO = True` for a quick smoke-test on synthetic data (seconds).  
Set `DEMO = False` (**default**) to run on the full How2Sign dataset.

> This cell **must run before any other project import** — it patches `training.config` at module level so all downstream model/data imports pick up the right values.

In [ ]:
DEMO = False   # True = synthetic data smoke-test | False = full How2Sign

import training.config as cfg

if DEMO:
    cfg.MAX_FRAMES       = 8
    cfg.IMG_SIZE         = 64
    cfg.D_MODEL          = 128
    cfg.N_HEADS          = 2
    cfg.N_ENC_LAYERS     = 2
    cfg.N_DEC_LAYERS     = 2
    cfg.DIM_FEEDFORWARD  = 256
    cfg.VOCAB_SIZE       = 100
    cfg.MAX_TEXT_LEN     = 16
    cfg.MICRO_BATCH_SIZE = 2
    cfg.GRAD_ACCUM_STEPS = 1
    cfg.NUM_WORKERS      = 0
    cfg.PHASE1_EPOCHS    = 2
    cfg.PHASE2_EPOCHS    = 1
    cfg.TOTAL_EPOCHS     = 3
    cfg.BEAM_SIZE        = 1
    cfg.TOKENIZER_MODEL  = ROOT / "data" / "tokenizer_demo.model"
    print("[DEMO] synthetic data, tiny model")
else:
    # Full config — only override what differs from defaults
    cfg.NUM_WORKERS = 4
    print("[FULL] How2Sign dataset, production model")

cfg.USE_AMP = DEVICE.type == "cuda"

# Convenient aliases used throughout the notebook
MAX_FRAMES   = cfg.MAX_FRAMES
IMG_SIZE     = cfg.IMG_SIZE
MAX_TEXT_LEN = cfg.MAX_TEXT_LEN
PAD_ID, BOS_ID, EOS_ID = cfg.PAD_ID, cfg.BOS_ID, cfg.EOS_ID

print(f"  MAX_FRAMES={MAX_FRAMES}  IMG_SIZE={IMG_SIZE}  D_MODEL={cfg.D_MODEL}")
print(f"  N_HEADS={cfg.N_HEADS}  ENC={cfg.N_ENC_LAYERS}  DEC={cfg.N_DEC_LAYERS}")
print(f"  VOCAB={cfg.VOCAB_SIZE}  MAX_TEXT={MAX_TEXT_LEN}  AMP={cfg.USE_AMP}")
print(f"  PHASE1={cfg.PHASE1_EPOCHS}ep @ lr={cfg.PHASE1_LR}  "
      f"PHASE2={cfg.PHASE2_EPOCHS}ep @ lr={cfg.PHASE2_LR}")

## 3. Data Pipeline Walkthrough

These cells illustrate the three-stage preprocessing pipeline on one example clip.  
In DEMO mode a synthetic MP4 is generated; in full mode we peek at one record from the train split.

### 3.1 Frame extraction

In [ ]:
import av
from data.download import extract_frames

def _make_synthetic_video(n_frames: int = 12, size: int = IMG_SIZE) -> bytes:
    buf = io.BytesIO()
    container = av.open(buf, mode="w", format="mp4")
    stream = container.add_stream("libx264", rate=25)
    stream.width = stream.height = size
    stream.pix_fmt = "yuv420p"
    stream.options = {"crf": "23"}
    rng = np.random.default_rng(0)
    for _ in range(n_frames):
        data  = rng.integers(0, 255, (size, size, 3), dtype=np.uint8)
        frame = av.VideoFrame.from_ndarray(data, format="rgb24")
        for pkt in stream.encode(frame): container.mux(pkt)
    for pkt in stream.encode(): container.mux(pkt)
    container.close()
    return buf.getvalue()

if DEMO:
    sample_video = _make_synthetic_video()
    print(f"Synthetic video : {len(sample_video):,} bytes")
else:
    from data.download import load_dataset_split
    _peek = next(iter(load_dataset_split("train", streaming=True)))
    _vbytes = _peek.get("mp4") or _peek.get("video") or b""
    sample_video = (_vbytes.get("bytes") or b"") if isinstance(_vbytes, dict) else _vbytes
    _text = (_peek.get("json") or {}).get("SENTENCE") or _peek.get("translation", "")
    print(f"Real train clip : {len(sample_video):,} bytes  "
          f"sentence: {_text[:60]!r}")

frames, frame_mask = extract_frames(sample_video, max_frames=MAX_FRAMES, img_size=IMG_SIZE)
IMEAN = torch.tensor(cfg.IMAGENET_MEAN).view(3,1,1)
ISTD  = torch.tensor(cfg.IMAGENET_STD).view(3,1,1)

print(f"frames     : {frames.shape}  ({frame_mask.sum().item()} real / {MAX_FRAMES} total)")

n_show = min(4, int(frame_mask.sum()))
fig, axes = plt.subplots(1, n_show, figsize=(3*n_show, 3))
if n_show == 1: axes = [axes]
for i, ax in enumerate(axes):
    ax.imshow((frames[i]*ISTD+IMEAN).clamp(0,1).permute(1,2,0).numpy())
    ax.set_title(f"Frame {i}")
    ax.axis("off")
plt.suptitle("Extracted frames (ImageNet-normalised → denormalised for display)")
plt.tight_layout(); plt.show()

### 3.2 Landmark extraction & heatmap

MediaPipe Holistic → 33 pose + 21 left-hand + 21 right-hand keypoints per frame.  
Weighted Gaussian blobs form the **spatial attention prior** fed into `SpatialAttention`.

> MediaPipe ≥ 0.10 removed the `solutions` API — the code returns zeros gracefully.

In [ ]:
from data.landmarks import extract_landmarks, create_landmark_heatmap, _MP_AVAILABLE

print(f"MediaPipe solutions API : {_MP_AVAILABLE}")
if not _MP_AVAILABLE:
    print("  → returning zero landmarks (heatmap is all-zero prior — model still trains fine)")

frame0_np = (( frames[0]*ISTD+IMEAN).clamp(0,1).permute(1,2,0).numpy()*255).astype(np.uint8)
landmarks = extract_landmarks(frame0_np)           # (75, 2)
heatmap   = create_landmark_heatmap(landmarks, img_size=IMG_SIZE)  # (H, W)

print(f"landmarks : {landmarks.shape}  non-zero={(landmarks.sum(1)>0).sum()}/75")
print(f"heatmap   : {heatmap.shape}   max={heatmap.max():.3f}")

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
axes[0].imshow(frame0_np); axes[0].set_title("Frame"); axes[0].axis("off")

axes[1].imshow(frame0_np)
vis = landmarks[landmarks.sum(1) > 0]
if len(vis):
    axes[1].scatter(vis[:,0]*(IMG_SIZE-1), vis[:,1]*(IMG_SIZE-1),
                    c="lime", s=10, linewidths=0)
axes[1].set_title(f"Landmarks ({len(vis)} detected)"); axes[1].axis("off")

axes[2].imshow(heatmap, cmap="hot")
axes[2].set_title("Heatmap attention prior"); axes[2].axis("off")

plt.tight_layout(); plt.show()

## 4. Tokenizer

SentencePiece BPE. Special IDs — PAD=0, BOS=1, EOS=2, UNK=3 — are fixed at training time.  
Full run trains on 50 K sentences from the train split; demo trains on 16 synthetic sentences.

In [ ]:
from data.tokenizer import SignTokenizer

tokenizer = SignTokenizer()

if DEMO:
    _demo_corpus = [
        "the cat sat on the mat", "she is signing hello to you",
        "please turn left at the corner", "good morning how are you today",
        "i would like some water please", "thank you very much for helping",
        "where is the nearest bathroom", "my name is alice nice to meet you",
        "the weather is nice today outside", "can you help me find the store",
        "i am learning sign language now", "see you later goodbye have a nice day",
        "what time is it please tell me", "i need to go home soon",
        "the quick brown fox jumps over", "nice to meet you welcome here",
    ]
    tokenizer.train(_demo_corpus * 20, model_path=cfg.TOKENIZER_MODEL,
                    vocab_size=cfg.VOCAB_SIZE)
elif cfg.TOKENIZER_MODEL.exists():
    tokenizer.load(cfg.TOKENIZER_MODEL)
    print(f"Tokenizer loaded from cache — vocab={tokenizer.vocab_size}")
else:
    print("Training tokenizer on 50 K train-split sentences (streaming)...")
    from data.download import load_dataset_split
    _corpus = [
        ex["translation"]
        for ex in load_dataset_split("train", streaming=True).take(50_000)
    ]
    tokenizer.train(_corpus, model_path=cfg.TOKENIZER_MODEL, vocab_size=cfg.VOCAB_SIZE)

print(f"Vocabulary size : {tokenizer.vocab_size}")

# Round-trip check
_s = "i am learning sign language"
_ids = tokenizer.encode(_s)
print(f"\n  encode({_s!r}) → {_ids}")
print(f"  decode       → {tokenizer.decode(_ids)!r}")

## 5. Dataset & DataLoader — All Three Splits

**train** → shuffled, used for gradient updates.  
**val** → unshuffled, monitored each epoch (no gradient).  
**test** → unshuffled, used **only** for final evaluation after training is complete.

In [ ]:
from data.dataset import How2SignDataset, collate_fn
from torch.utils.data import DataLoader

if DEMO:
    _vb = _make_synthetic_video()
    _all = [{"video": _vb, "translation": _demo_corpus[i % len(_demo_corpus)]}
            for i in range(24)]
    train_records, val_records, test_records = _all[:16], _all[16:20], _all[20:]
else:
    from data.download import load_dataset_split
    print("Downloading How2Sign — train split (this can take several minutes)...")
    train_records = list(load_dataset_split("train", streaming=False))
    print(f"  train : {len(train_records):,} clips")

    print("Downloading How2Sign — val split...")
    val_records = list(load_dataset_split("val", streaming=False))
    print(f"  val   : {len(val_records):,} clips")

    print("Downloading How2Sign — test split...")
    test_records = list(load_dataset_split("test", streaming=False))
    print(f"  test  : {len(test_records):,} clips")

# Build datasets — extract_lm=True in full mode so MediaPipe runs (if available)
_lm = not DEMO
train_ds = How2SignDataset(train_records, tokenizer,
                           max_frames=MAX_FRAMES, img_size=IMG_SIZE,
                           max_text_len=MAX_TEXT_LEN, extract_lm=_lm)
val_ds   = How2SignDataset(val_records,   tokenizer,
                           max_frames=MAX_FRAMES, img_size=IMG_SIZE,
                           max_text_len=MAX_TEXT_LEN, extract_lm=_lm)
test_ds  = How2SignDataset(test_records,  tokenizer,
                           max_frames=MAX_FRAMES, img_size=IMG_SIZE,
                           max_text_len=MAX_TEXT_LEN, extract_lm=_lm)

_bs = cfg.MICRO_BATCH_SIZE
_nw = cfg.NUM_WORKERS
_pm = not DEMO  # pin_memory only useful with CUDA + workers

train_loader = DataLoader(train_ds, batch_size=_bs,   shuffle=True,
                          collate_fn=collate_fn, num_workers=_nw, pin_memory=_pm)
val_loader   = DataLoader(val_ds,   batch_size=_bs*2, shuffle=False,
                          collate_fn=collate_fn, num_workers=_nw, pin_memory=_pm)
test_loader  = DataLoader(test_ds,  batch_size=_bs*2, shuffle=False,
                          collate_fn=collate_fn, num_workers=_nw, pin_memory=_pm)

print(f"\nDataLoaders ready:")
print(f"  train : {len(train_ds):>6,} samples  {len(train_loader):>5,} batches")
print(f"  val   : {len(val_ds):>6,} samples  {len(val_loader):>5,} batches")
print(f"  test  : {len(test_ds):>6,} samples  {len(test_loader):>5,} batches")

In [ ]:
# Inspect one batch to confirm shapes
batch = next(iter(train_loader))
print("Train batch shapes:")
for k, v in batch.items():
    print(f"  {k:<12}: {v.shape if hasattr(v,'shape') else v}")

# Split-size bar chart
fig, ax = plt.subplots(figsize=(5, 3))
names = ["train", "val", "test"]
sizes = [len(train_ds), len(val_ds), len(test_ds)]
colors = ["#4C72B0", "#DD8452", "#55A868"]
bars = ax.bar(names, sizes, color=colors, width=0.5)
for bar, s in zip(bars, sizes):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(sizes)*0.01,
            f"{s:,}", ha="center", va="bottom", fontsize=10)
ax.set_ylabel("Clips")
ax.set_title("How2Sign split sizes")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

## 6. Model Architecture

```
Video  (B, T, 3, H, W)
  └─ VisualEncoder
       ResNet-18 (frozen ph.1 / unfrozen ph.2)
       + SpatialAttention(heatmap prior)        → (B, T, d_model)
  └─ TemporalEncoder
       Sinusoidal PE + 4-layer Transformer Enc  → (B, T, d_model)
  └─ TextDecoder
       Embedding + 4-layer Transformer Dec
       + causal mask + beam search              → (B, S, vocab_size)
```

In [ ]:
from models.translator import SignLanguageTranslator

model = SignLanguageTranslator(
    vocab_size=tokenizer.vocab_size,
    d_model=cfg.D_MODEL,
    freeze_backbone=True,
).to(DEVICE)

params = model.count_parameters()
print("Parameters:")
for k, n in params.items():
    print(f"  {k:<20}: {n:>12,}")

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n  Trainable : {trainable:>12,}  ({100*trainable/total:.1f}%)")
print(f"  Frozen    : {total-trainable:>12,}  (ResNet backbone, phase 1)")

In [ ]:
# Pie chart + forward-pass sanity check
labels = [k for k in params if k != "total"]
sizes  = [params[k] for k in labels]
colors = ["#4C72B0", "#DD8452", "#55A868"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.pie(sizes, labels=labels, autopct="%1.1f%%", colors=colors, startangle=140)
ax1.set_title("Parameter share")
ax2.barh(labels, [s/1e6 for s in sizes], color=colors)
ax2.set_xlabel("Parameters (M)")
ax2.set_title("Parameters per module")
for i, v in enumerate([s/1e6 for s in sizes]):
    ax2.text(v+max(s/1e6 for s in sizes)*0.01, i, f"{v:.2f}M", va="center")
plt.tight_layout(); plt.show()

# Forward-pass check
model.eval()
with torch.no_grad():
    _tb = next(iter(train_loader))
    _f  = _tb["frames"].to(DEVICE)
    _h  = _tb["heatmaps"].to(DEVICE)
    _t  = _tb["token_ids"].to(DEVICE)
    _fm = _tb["frame_mask"].to(DEVICE)
    _ti = _t[:, :-1]
    _logits = model(_f, _ti, heatmaps=_h, frame_mask=_fm,
                    tgt_key_padding_mask=(_ti==PAD_ID))
print(f"\nForward pass: frames={_f.shape} → logits={_logits.shape}  ✓")

## 7. Training

Uses the `Trainer` class from `training/train.py`.  
- Trains on **train** split  
- Monitors BLEU-4 on **val** split each epoch  
- Saves best-val-BLEU checkpoint automatically

In [ ]:
from training.train import Trainer

trainer = Trainer(
    model, tokenizer,
    train_loader,
    val_loader,
    test_loader,           # held out — only evaluated after training
    use_wandb=False,       # set True if WANDB_API_KEY is configured
    device=DEVICE,
)

# ── Optional: resume from a saved checkpoint ─────────────────────────────────
# RESUME = cfg.CHECKPOINT_DIR / "checkpoint_best.pt"
# if RESUME.exists():
#     trainer.load_checkpoint(RESUME)
#     print(f"Resumed from {RESUME}")

print(f"Ready to train for {cfg.TOTAL_EPOCHS} epochs "
      f"({cfg.PHASE1_EPOCHS} frozen + {cfg.PHASE2_EPOCHS} unfrozen).")
print(f"Effective batch size : {cfg.MICRO_BATCH_SIZE * cfg.GRAD_ACCUM_STEPS}")
print(f"Checkpoints → {cfg.CHECKPOINT_DIR}")

In [ ]:
# ── Run training ─────────────────────────────────────────────────────────────
# trainer.train() runs phases 1+2 and then auto-evaluates on the test split.
# Progress is printed to stdout; checkpoints are saved every CHECKPOINT_EVERY epochs.

import logging
logging.getLogger().setLevel(logging.INFO)   # show training progress

t0 = time.time()
trainer.train()
elapsed = time.time() - t0

logging.getLogger().setLevel(logging.WARNING)
print(f"\nTotal training time : {elapsed/3600:.2f} h")

In [ ]:
# ── Training curve ────────────────────────────────────────────────────────────
# Read the checkpoint history to plot loss + LR.
# If you want richer logging use WandB (set use_wandb=True above).

import glob
ckpt_files = sorted(glob.glob(str(cfg.CHECKPOINT_DIR / "checkpoint_epoch*.pt")))
print(f"Checkpoints found : {len(ckpt_files)}")
for f in ckpt_files:
    ck = torch.load(f, map_location="cpu")
    print(f"  {Path(f).name}  epoch={ck['epoch']+1}  loss={ck['loss']:.4f}")

## 8. Test-Set Evaluation

The **test split was never used during training or hyperparameter selection**.  
These are the headline numbers for your paper / report.

Metrics:
- **BLEU-4** — corpus-level n-gram precision (primary)
- **ROUGE-L** — longest common subsequence recall  
- **METEOR** — synonym-aware precision / recall

In [ ]:
# Load the best checkpoint before running test evaluation
best_ckpt = cfg.CHECKPOINT_DIR / "checkpoint_best.pt"
if best_ckpt.exists():
    ckpt = torch.load(best_ckpt, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"Loaded best checkpoint (epoch {ckpt['epoch']+1}, "
          f"best_bleu={ckpt.get('best_bleu',0):.2f})")
else:
    print("No best checkpoint found — using current model weights.")

In [ ]:
import logging
logging.getLogger().setLevel(logging.INFO)

test_metrics, test_hyps, test_refs = trainer.evaluate(
    test_loader, split="test", beam_size=cfg.BEAM_SIZE
)

logging.getLogger().setLevel(logging.WARNING)

In [ ]:
# Show 10 example predictions from test set
print(f"{'REF':<55}  HYP")
print("-" * 110)
for ref, hyp in zip(test_refs[:10], test_hyps[:10]):
    print(f"{ref[:55]:<55}  {hyp[:55]}")

# Metric bar chart
m_keys   = ["bleu4", "rougeL_f1", "meteor"]
m_labels = ["BLEU-4", "ROUGE-L", "METEOR"]
m_vals   = [test_metrics.get(k, 0.0) for k in m_keys]

fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.bar(m_labels, m_vals, color=["#4C72B0","#DD8452","#55A868"], width=0.5)
for bar, v in zip(bars, m_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(m_vals)*0.02,
            f"{v:.2f}", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_ylim(0, max(max(m_vals)*1.25, 1.0))
ax.set_ylabel("Score")
ax.set_title(f"Test-Set Results  (n={len(test_refs):,})")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Hypothesis-length distribution
hyp_lens = [len(h.split()) for h in test_hyps]
ref_lens = [len(r.split()) for r in test_refs]

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(ref_lens, bins=30, alpha=0.6, label=f"Reference (μ={np.mean(ref_lens):.1f})", color="steelblue")
ax.hist(hyp_lens, bins=30, alpha=0.6, label=f"Hypothesis (μ={np.mean(hyp_lens):.1f})", color="darkorange")
ax.set_xlabel("Sentence length (words)")
ax.set_ylabel("Count")
ax.set_title("Test-set length distribution")
ax.legend()
plt.tight_layout(); plt.show()

## 9. LLM-as-Judge on Test Predictions

Claude scores each hypothesis on three 1–5 dimensions and we compute Pearson correlation vs BLEU.  
Requires `ANTHROPIC_API_KEY`.

In [ ]:
import os
API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
RUN_LLM_JUDGE = bool(API_KEY)
if RUN_LLM_JUDGE:
    print(f"API key found ({API_KEY[:12]}...) — LLM judge will run on test predictions.")
else:
    print("ANTHROPIC_API_KEY not set — skipping LLM judge.")
    print("  os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'")

In [ ]:
if RUN_LLM_JUDGE:
    from evaluation.llm_judge import (
        judge_batch, summarise, print_examples,
        pearson_correlation, per_sample_bleu, save_scores,
    )

    print(f"Scoring {min(len(test_hyps), cfg.LLM_JUDGE_SAMPLES)} test samples "
          f"with {cfg.LLM_JUDGE_MODEL}...")
    llm_scores = judge_batch(
        test_hyps, test_refs,
        model=cfg.LLM_JUDGE_MODEL,
        batch_size=cfg.LLM_JUDGE_BATCH_SIZE,
        max_samples=cfg.LLM_JUDGE_SAMPLES,
    )

    llm_summary = summarise(llm_scores)
    print("\nLLM Judge Summary (test split):")
    for k, v in llm_summary.items():
        print(f"  {k:<22}: {v:.3f}")

    sample_bleu = per_sample_bleu(test_hyps[:len(llm_scores)], test_refs[:len(llm_scores)])
    r = pearson_correlation(llm_scores, sample_bleu)
    print(f"\n  Pearson r(LLM composite, BLEU-4) = {r:.4f}")

    save_scores(llm_scores, str(cfg.CHECKPOINT_DIR / "llm_judge_test.json"))
    print_examples(llm_scores, n=10)
else:
    print("LLM judge skipped.")

In [ ]:
if RUN_LLM_JUDGE and llm_scores:
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, attr, lbl in zip(axes,
                              ["adequacy", "fluency", "meaning"],
                              ["Adequacy", "Fluency", "Meaning"]):
        vals = [getattr(s, attr) for s in llm_scores]
        ax.hist(vals, bins=[.5,1.5,2.5,3.5,4.5,5.5], color="steelblue", edgecolor="white")
        mu = np.mean(vals)
        ax.axvline(mu, color="orange", ls="--", label=f"μ={mu:.2f}")
        ax.set_title(lbl); ax.set_xlabel("Score (1–5)"); ax.set_xticks([1,2,3,4,5])
        ax.legend()
    plt.suptitle("LLM Judge — Test Split Score Distributions", fontsize=13)
    plt.tight_layout(); plt.show()

## 10. Ablation Study on Test Split

Quantifies the contribution of **landmark-aware spatial attention** by comparing three conditions.

| Config | What changes |
|---|---|
| `full_model` | Real MediaPipe heatmaps fed to SpatialAttention |
| `no_spatial_attn` | `heatmaps=None` — attention gate fully bypassed |
| `zero_heatmaps` | All-zero heatmap — attention gate active but uninformative |

In [ ]:
from evaluation.ablation import run_ablation, print_ablation_table

ablation_results = run_ablation(
    model_full=model,
    tokenizer=tokenizer,
    test_loader=test_loader,   # always run ablation on the test split
    device=DEVICE,
    beam_size=cfg.BEAM_SIZE,
)
print_ablation_table(ablation_results)

In [ ]:
cfg_names  = list(ablation_results.keys())
b_vals = [ablation_results[c].get("bleu4",     0) for c in cfg_names]
r_vals = [ablation_results[c].get("rougeL_f1", 0) for c in cfg_names]
m_vals = [ablation_results[c].get("meteor",    0) for c in cfg_names]

x, w = np.arange(len(cfg_names)), 0.25
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x-w, b_vals, w, label="BLEU-4",    color="#4C72B0")
ax.bar(x,   r_vals, w, label="ROUGE-L",   color="#DD8452")
ax.bar(x+w, m_vals, w, label="METEOR",    color="#55A868")
ax.set_xticks(x); ax.set_xticklabels(cfg_names, rotation=10)
ax.set_ylabel("Score")
ax.set_title("Ablation — Effect of Spatial Attention (test split)")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

if "full_model" in ablation_results and "no_spatial_attn" in ablation_results:
    delta = ablation_results["full_model"].get("bleu4",0) - \
            ablation_results["no_spatial_attn"].get("bleu4",0)
    print(f"\nSpatial attention contribution : {delta:+.2f} BLEU-4 points")

## 11. Save & Load Checkpoint

In [ ]:
CKPT_PATH = cfg.CHECKPOINT_DIR / "checkpoint_final.pt"
cfg.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

torch.save({
    "model_state_dict": model.state_dict(),
    "vocab_size":       tokenizer.vocab_size,
    "d_model":          cfg.D_MODEL,
    "test_metrics":     test_metrics,
}, CKPT_PATH)
print(f"Saved  → {CKPT_PATH}  ({CKPT_PATH.stat().st_size/1e6:.1f} MB)")

In [ ]:
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Loaded ← {CKPT_PATH}")
print(f"  vocab={ckpt['vocab_size']}  d_model={ckpt['d_model']}")
if "test_metrics" in ckpt:
    tm = ckpt["test_metrics"]
    print(f"  test BLEU-4={tm.get('bleu4',0):.2f}  "
          f"ROUGE-L={tm.get('rougeL_f1',0):.4f}  "
          f"METEOR={tm.get('meteor',0):.4f}")

## 12. Interactive Inference

Pass any MP4 byte-string to `translate_video` to get an English translation.

In [ ]:
def translate_video(video_bytes: bytes, beam_size: int = cfg.BEAM_SIZE) -> str:
    """End-to-end: raw MP4 bytes → English string."""
    frames_v, mask_v = extract_frames(video_bytes, max_frames=MAX_FRAMES, img_size=IMG_SIZE)
    frames_v = frames_v.unsqueeze(0).to(DEVICE)    # (1, T, 3, H, W)
    mask_v   = mask_v.unsqueeze(0).to(DEVICE)      # (1, T)
    hm_v     = torch.zeros(1, MAX_FRAMES, IMG_SIZE, IMG_SIZE, device=DEVICE)
    model.eval()
    with torch.no_grad():
        return model.translate(frames_v, tokenizer,
                               heatmaps=hm_v, frame_mask=mask_v,
                               beam_size=beam_size)[0]


# Try on the first test clip
_test_clip = test_records[0]
_clip_bytes = _test_clip.get("video", b"")
if isinstance(_clip_bytes, dict):
    _clip_bytes = _clip_bytes.get("bytes", b"")

pred = translate_video(_clip_bytes)
ref  = _test_clip.get("translation", "")

print(f"REF  : {ref}")
print(f"PRED : {pred}")

---

## Module Reference

| Step | File | Key API |
|---|---|---|
| Frame extraction | `data/download.py` | `extract_frames(bytes, max_frames, img_size)` |
| Landmark detection | `data/landmarks.py` | `extract_landmarks(frame_rgb)` |
| Heatmap | `data/landmarks.py` | `create_landmark_heatmap(landmarks, img_size)` |
| Tokenizer | `data/tokenizer.py` | `SignTokenizer.train / load / encode / decode` |
| Dataset | `data/dataset.py` | `How2SignDataset`, `collate_fn` |
| Data splits | `data/download.py` | `load_dataset_split(split)` — `'train'/'val'/'test'` |
| Visual encoder | `models/visual_encoder.py` | `VisualEncoder` (ResNet-18 + SpatialAttention) |
| Temporal encoder | `models/temporal_encoder.py` | `TemporalEncoder` (Transformer Encoder) |
| Text decoder | `models/text_decoder.py` | `TextDecoder` (Transformer Decoder + beam search) |
| Full model | `models/translator.py` | `SignLanguageTranslator.forward / .translate` |
| Training | `training/train.py` | `Trainer(model, tok, train_loader, val_loader, test_loader)` |
| Evaluation | `training/train.py` | `Trainer.evaluate(loader, split)` |
| Config | `training/config.py` | all hyperparameters |
| Metrics | `evaluation/metrics.py` | `compute_all`, `print_results` |
| LLM judge | `evaluation/llm_judge.py` | `judge_batch`, `summarise`, `save_scores` |
| Ablation | `evaluation/ablation.py` | `run_ablation(model, tok, test_loader, device)` |